<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L03-a-mini-arena/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L03-a-mini-arena/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T10-L03-a-mini-arena/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T10-L03-a-mini-arena/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T10-L03 · A mini arena: from pairwise votes to Bradley-Terry ratings with honest confidence intervals

**You will build:** the statistics behind a pairwise-vote leaderboard: a win-count matrix, a
maximum-likelihood Bradley-Terry fit, a fit that also models "voters favour the left-hand
answer", a bootstrap over votes, and the measurement that says how many votes it takes
before two neighbouring systems stop swapping places.

**Time:** ~50 minutes · **Runs on:** a laptop CPU, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track (the habit of printing what you measured, and
never typing it).

There are no people and no language models in this lesson. Votes are simulated from
strengths that YOU can see, with a position bias and a fraction of careless voters planted
on purpose, because a leaderboard method is only trustworthy if it can recover what was
planted, and on real votes nobody knows the truth to check against.

By the end you will be able to:

1. Implement a win-count matrix from pairwise votes and a maximum-likelihood Bradley-Terry
   fit by fixed-point iteration, with a pseudo-count that keeps an undefeated system finite.
2. Implement a Bradley-Terry fit with a position-bias term, and measure how much an
   unrandomised side assignment distorts the ratings with and without it.
3. Implement a bootstrap over votes that returns rating replicates, and turn them into
   intervals and into the share of replicates in which two neighbours swap places.
4. Measure how many votes each adjacent pair needs before its order stops flipping, and stays
   stopped, on simulated vote streams.
5. Measure how a fraction of random voters compresses the rating scale.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write the logistic (sigmoid) function and say what it does to a difference of two scores
- say what maximum likelihood estimation means, and read a log-likelihood
- explain what a gradient and a Hessian are, and how one Newton step uses them
- write numpy array expressions with broadcasting, a matrix transpose product and np.linalg.solve

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Chatbot Arena: An Open Platform for Evaluating LLMs by Human Preference](https://arxiv.org/abs/2403.04132) · Chiang et al. (arXiv) · *free* — section 4, From Pairwise Comparisons to Rankings: the Bradley-Terry fit, confidence intervals by bootstrap or sandwich errors, and approximate ranks that allow for that uncertainty
- [Speech and Language Processing (3rd ed. draft), chapter 4: Logistic Regression and Text Classification](https://web.stanford.edu/~jurafsky/slp3/4.pdf) · Dan Jurafsky and James H. Martin (Stanford; free draft) · *free* — 4.2 the sigmoid, 4.5 the likelihood-based cross-entropy loss and 4.6 its gradient: the position-bias model of exercise 3 is exactly a logistic regression
- [Convex Optimization, section 9.5 (Newton's method)](https://web.stanford.edu/~boyd/cvxbook/) · Stephen Boyd and Lieven Vandenberghe (Cambridge University Press; free PDF from the authors) · *free* — the Newton step, built from the gradient and the Hessian, that fit_bt_biased takes for its logistic loss
- [Computational and Inferential Thinking, section 13.2 (the bootstrap)](https://inferentialthinking.com/chapters/13/2/bootstrap/index.html) · Ani Adhikari, John DeNero and David Wagner · *free* — resampling the data with replacement and reading percentile intervals off the replicates, the idea behind bootstrap_theta
- [An Introduction to the Bootstrap](https://www.routledge.com/An-Introduction-to-the-Bootstrap/Efron-Tibshirani/p/book/9780412042317) · Bradley Efron and R. J. Tibshirani (Chapman & Hall/CRC) · *paid* — the standard text on the bootstrap; chapter 6 (bootstrap standard errors) and chapter 13 (percentile intervals) are the theory behind the rating intervals and swap shares

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import os
import signal
import sys
import threading
import time
import traceback
from typing import Callable, NamedTuple, Sequence

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20260929          # this lesson's one fixed seed: every vote, every voter and every
                          # bootstrap resample derives from it, so re-running prints the same
                          # numbers on any machine.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("win_counts",),
    "exercise 2": ("fit_bt",),
    "exercise 3": ("fit_bt_biased",),
    "exercise 4": ("bootstrap_theta",),
    "exercise 5": ("votes_needed",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (fit_bt_biased)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# A check that never returns would freeze the notebook (and the grader, which imports it). Every
# check below runs under a time limit far above what a correct answer needs, and so does every
# single fit a demo asks your code for; a whole demo gets a much longer limit, because slow but
# correct code is allowed to be slow.
_CHECK_SECONDS = 2
_CALL_SECONDS = 10
_DEMO_SECONDS = 90


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that whoever started us had already set
    keeps its own deadline if it is due first, and is put back, with the time it had left and
    its repeat interval, when the block ends. A correct answer finishes far inside the limit,
    and nothing that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose "
               "stopping condition is never met (a counter that is never "
               "advanced, an index that wraps round, a tolerance that is never "
               "reached exactly) runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. What an arena does, and the arena we will simulate

An arena shows a voter two anonymous answers, one on the left (slot A) and one on the right
(slot B), and records which one they preferred. Nobody scores anything on a scale; the raw
data is a long list of triples `(system in slot A, system in slot B, did A win)`. Turning
that list into a leaderboard is a statistics problem, and this lesson is that problem.

The arena below is simulated so that the truth is known. Three things are planted:

* every system has a **strength**; the chance that A beats B, for a careful voter, is a
  logistic function of the difference of their strengths;
* a **position bias** `beta`: a bonus, on the same logistic scale, that goes to whichever
  system happens to sit in slot A, whatever its quality;
* a fraction `p_random` of votes come from a voter who is not paying attention, and picks a
  side by a fair coin.

As of 29 September 2026 the Chatbot Arena paper models each battle as a pair of models and
a binary winner, fits Bradley-Terry coefficients to them by maximum likelihood, and, in its
automatic benchmark, swaps which slot a baseline answer sits in "to avoid positional bias".
`claims.yaml` has the quotes. Nothing in this lesson touches a network or a real person's
vote.

`simulate_votes` is given to you. Read it: every later exercise is about undoing what it
does.

In [ ]:
NAMES = ["Aster", "Birch", "Cedar", "Dune", "Ember", "Fjord", "Grove", "Heath"]
K = len(NAMES)
# Planted strengths, listed in the order the systems were "released" (index 0 is the oldest).
# Newer systems are usually, not always, stronger. Only differences matter, so they are
# centred to have mean zero.
_RAW_STRENGTH = np.array([0.90, 0.00, 1.35, 0.60, 1.95, 1.20, 2.03, 1.75])
THETA_TRUE = _RAW_STRENGTH - _RAW_STRENGTH.mean()
PLANTED_BETA = 0.35       # position bias, on the logit scale
PLANTED_RANDOM = 0.15     # share of votes cast by a voter who flips a coin


def to_elo(theta):
    """Display ratings on an Elo-like scale: 1000 + 400 * theta / ln(10). A display choice
    only: ratings are meaningful as DIFFERENCES, and any shift or stretch of the axis leaves
    every predicted win probability where it was."""
    return 1000.0 + 400.0 * np.asarray(theta) / np.log(10.0)


def simulate_votes(theta, n_votes, rng, beta=0.0, p_random=0.0, randomise_sides=True):
    """Simulate `n_votes` pairwise votes between systems with strengths `theta`. Given to you.

    Returns three integer arrays `(a, b, a_won)`: the system in slot A, the system in slot B,
    and 1 if the voter picked slot A, else 0. Each vote draws two different systems uniformly.
    If `randomise_sides` the slots are assigned by a fair coin; otherwise the OLDER system
    (lower index) is always in slot A, which is what a lazy collector does by default.
    A careful voter picks A with probability sigmoid(theta_A - theta_B + beta); with
    probability `p_random` the voter ignores everything and picks A with probability 1/2.
    """
    k = len(theta)
    i = rng.integers(0, k, n_votes)
    j = (i + rng.integers(1, k, n_votes)) % k
    swap = rng.random(n_votes) < 0.5
    if randomise_sides:
        a, b = np.where(swap, j, i), np.where(swap, i, j)
    else:
        a, b = np.minimum(i, j), np.maximum(i, j)
    p_a = 1.0 / (1.0 + np.exp(-(theta[a] - theta[b] + beta)))
    p_a = np.where(rng.random(n_votes) < p_random, 0.5, p_a)
    a_won = (rng.random(n_votes) < p_a).astype(np.int64)
    return a, b, a_won


print(f"{K} systems, planted strengths (Elo-like display; only differences matter):")
for _idx in np.argsort(THETA_TRUE):
    print(f"  {NAMES[_idx]:<6} released #{_idx}  planted rating {to_elo(THETA_TRUE)[_idx]:7.1f}")

_rng = np.random.default_rng(SEED)
_a, _b, _y = simulate_votes(THETA_TRUE, 6, _rng, PLANTED_BETA, PLANTED_RANDOM)
print("\nthe first few votes:")
for _x, _z, _w in zip(_a, _b, _y):
    print(f"  A = {NAMES[_x]:<6} B = {NAMES[_z]:<6} -> the voter picked {'A' if _w else 'B'}")

# How visible is the position bias in the raw votes? Same arena, same voters, only beta differs.
for _label, _beta in (("no position bias", 0.0), ("planted position bias", PLANTED_BETA)):
    _a, _b, _y = simulate_votes(THETA_TRUE, 40_000, np.random.default_rng(SEED + 1),
                                _beta, PLANTED_RANDOM)
    print(f"share of votes won by the system in slot A, {_label}: {_y.mean():.3f}")

## 2. From votes to a win matrix, and the model that explains it

The Bradley-Terry model says that every system `i` has one number `theta_i`, and that

`P(i beats j) = 1 / (1 + exp(-(theta_i - theta_j)))`.

Two consequences drive the rest of the lesson. First, only differences matter: add the same
constant to every `theta` and no probability changes, so a rating is not an absolute
quantity and we pin the freedom down by centring (mean zero). Second, ratings live on the
logit scale, so equal gaps mean equal odds ratios, and an "Elo" number is just this scale
stretched and shifted for display.

The next cell prints the win probabilities the planted strengths imply between every pair
of systems. Read a row as "this system beats that column with this probability". The
neighbours in rank order are far from certain, and no fitting method can make a coin-flip
matchup decisive: it can only say how uncertain it is. That gap is why the rest of this
lesson is about intervals.

In [ ]:
_order = list(np.argsort(-THETA_TRUE))
_p_true = 1.0 / (1.0 + np.exp(-(THETA_TRUE[:, None] - THETA_TRUE[None, :])))
print("P(row beats column) under the planted strengths, strongest first")
print("        " + " ".join(f"{NAMES[j][:5]:>5}" for j in _order))
for _i in _order:
    print(f"{NAMES[_i]:<7} " + " ".join("    -" if _i == _j else f"{_p_true[_i, _j]:5.2f}"
                                          for _j in _order))

## 3. Exercise 1 — `win_counts()`

The fitting code will not read the raw list of votes. It reads a **win matrix** `W`, where
`W[i, j]` is the number of votes in which system `i` beat system `j`, whichever slot each
was in. The row is always the winner. The diagonal is always zero, because a system never
plays itself. The slot only matters for the bias fit in exercise 3; here it is thrown away.

<details><summary>💡 Hint 1 — what to think about</summary>

`a_won` is about the SLOT, not about a system. For each vote, first work out which system
won and which lost; only then does a vote say anything about a row and a column of `W`.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build a winner array and a loser array by choosing, vote by vote, between the A-slot and
the B-slot system according to `a_won`. Then count each (winner, loser) pair: one way is to
flatten each pair into a single index and count occurrences of every index, then reshape;
another is a plain loop that increments one cell per vote.
</details>

In [ ]:
def win_counts(a, b, a_won, k):
    """Count, for every ordered pair of systems, how often the first beat the second.

    `a` and `b` are integer arrays holding the system in slot A and slot B of each vote,
    `a_won` is 1 where the voter picked slot A and 0 where they picked slot B, and `k` is the
    number of systems.

    Example:
        >>> win_counts([0, 1], [1, 0], [1, 1], 2)
        array([[0, 1],
               [1, 0]])
        (system 0 won as A in the first vote, system 1 won as A in the second)

    Returns:
        a (k, k) array whose entry [i, j] is the number of votes system i won against j.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_win_counts() -> None:
    a, b, y = np.array([0, 1, 2, 0]), np.array([1, 2, 0, 2]), np.array([1, 0, 1, 0])
    w = np.asarray(win_counts(a, b, y, 3))
    assert w.shape == (3, 3), f"the matrix has shape {w.shape}, it should be (k, k) = (3, 3)"
    assert w.sum() == len(y), (
        f"the matrix counts {w.sum()} wins from {len(y)} votes: every vote has exactly one winner"
    )
    assert np.array_equal(w, np.array([[0, 1, 0], [0, 0, 0], [2, 1, 0]])), (
        f"got {w.tolist()}. W[i, j] counts votes that system i WON against j. "
        "Remember a_won == 1 means the system in slot A won and 0 means slot B won; "
        "if your matrix looks like the transpose of what you expected, the row is the loser."
    )
    print("exercise 1 looks right")


_try("exercise 1", _check_win_counts)

## 4. Exercise 2 — `fit_bt()`: maximum likelihood by fixed-point iteration

Given `W`, the likelihood of the votes under Bradley-Terry is a product over pairs, and it
has one maximum. Write `p_i = exp(theta_i)` and let `w_i` be system `i`'s total wins and
`n_ij = W[i, j] + W[j, i]` the number of times `i` and `j` met. At the maximum, every system
has exactly the wins the model expects it to have:

`w_i = sum over j of n_ij * p_i / (p_i + p_j)`.

Rearranged as `p_i = w_i / sum over j of n_ij / (p_i + p_j)`, that is an update rule: plug
in the current `p`, get a new `p`, repeat. It is a classical fixed-point iteration for this
model, it never leaves the positive numbers, and each sweep can only raise the likelihood.
Because only differences matter, renormalise after every sweep so the `p` cannot drift.

One thing the maximum-likelihood recipe cannot handle: a system with **zero wins** wants
`p = 0`, and a system that **never loses** wants `p = infinity`. Early in an arena that
happens all the time. The fix used here is a `prior` pseudo-count: before fitting, add
`prior` to every off-diagonal entry of `W`, as if every pair had already met and split.

<details><summary>💡 Hint 1 — what to think about</summary>

Start every `p` at one. The sum in the denominator runs over all the OTHER systems, and
`n_ij` is symmetric in `i` and `j`. What must be true of the diagonal of `W` before you use
it, once the pseudo-count is in?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Add the pseudo-count off the diagonal only. Then loop: compute every system's denominator
from the current strengths in one array expression, divide the wins by it, divide out the
geometric mean so the strengths cannot drift, and stop when the log-strengths stop moving
(or after a generous cap on the iterations). Return the natural logarithm of the strengths,
shifted to mean zero.
</details>

In [ ]:
def fit_bt(W, prior=0.1, n_iter=2000, tol=1e-10):
    """Maximum-likelihood Bradley-Terry ratings from a win matrix.

    `W[i, j]` is how often system i beat j. Every off-diagonal entry gets `prior` added first
    (a pseudo-count, so an undefeated system stays finite). Iterate the update
    p_i = wins_i / sum_j n_ij / (p_i + p_j), renormalising the p to geometric mean 1 each
    sweep, until the largest change in any log p is below `tol` or `n_iter` sweeps have run.

    Example:
        >>> fit_bt([[0, 3], [1, 0]], prior=0).round(3)
        array([ 0.549, -0.549])
        (system 0 won three times to one, so p_0 / p_1 = 3 and theta_0 - theta_1 = ln 3)

    Returns:
        a length-k array of ratings theta = log p, on the natural-log scale, with mean zero.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_fit_bt() -> None:
    theta = np.asarray(fit_bt([[0, 6], [3, 0]], prior=0))
    assert theta.shape == (2,), f"expected one rating per system, got shape {theta.shape}"
    assert abs(theta[0] - theta[1] - np.log(2.0)) < 1e-6, (
        f"with two systems the maximum is where p_0 / p_1 equals wins_0 / wins_1; your gap is "
        f"{theta[0] - theta[1]:.4f} and ln 2 is {np.log(2.0):.4f}. Check the update rule, and "
        "that you return natural-log strengths (log p), not p itself"
    )
    assert abs(theta.sum()) < 1e-9, "ratings must be centred: they should sum to zero"
    w3 = np.array([[0, 8, 5], [4, 0, 6], [3, 2, 0]], dtype=float)
    th = np.asarray(fit_bt(w3, prior=0))
    p = np.exp(th)
    n3 = w3 + w3.T
    expected_wins = np.array([sum(n3[i, j] * p[i] / (p[i] + p[j]) for j in range(3) if j != i)
                              for i in range(3)])
    # p is only defined up to a constant, and the expected wins do not depend on that constant
    assert np.allclose(expected_wins, w3.sum(axis=1), atol=1e-4), (
        f"at the maximum every system's expected wins equal its observed wins; yours are "
        f"{np.round(expected_wins, 3).tolist()} against {w3.sum(axis=1).tolist()}. "
        "Iterate until the ratings stop moving, and sum over the OTHER systems only."
    )
    lone = np.asarray(fit_bt([[0, 4], [0, 0]]))
    assert np.all(np.isfinite(lone)) and lone[0] > lone[1], (
        f"a system that never lost must still get a finite rating above its opponent's, got "
        f"{lone.tolist()}: add `prior` to every off-diagonal entry BEFORE fitting"
    )
    weak = np.abs(np.asarray(fit_bt([[0, 4], [1, 0]], prior=0.1)))
    strong = np.abs(np.asarray(fit_bt([[0, 4], [1, 0]], prior=2.0)))
    assert strong.max() < weak.max(), (
        "a bigger pseudo-count should pull the ratings closer together; is `prior` actually "
        "used, and added to every off-diagonal pair?"
    )
    # counts from this lesson's own simulator, default pseudo-count: the fit must stop on these too
    rng = np.random.default_rng(17)
    for n_votes in (100, 400, 1600, 6400) * 5:
        a, b, y = simulate_votes(THETA_TRUE, n_votes, rng, PLANTED_BETA, PLANTED_RANDOM)
        sim = np.zeros((K, K))
        np.add.at(sim, (np.where(y == 1, a, b), np.where(y == 1, b, a)), 1.0)
        p = np.exp(np.asarray(fit_bt(sim)))
        aug = sim + 0.1 * (1.0 - np.eye(K))
        met = aug + aug.T
        assert np.allclose((met * p[:, None] / (p[:, None] + p[None, :])).sum(axis=1),
                           aug.sum(axis=1), atol=1e-4), (
            f"on {n_votes} simulated votes the expected wins do not match the observed ones (the "
            "pseudo-count included): iterate until the change is below `tol`"
        )
    spread = np.array([-3.0, 0.0, 3.0])
    counts = np.array([[500.0 / (1.0 + np.exp(-(spread[i] - spread[j]))) if i != j else 0.0
                        for j in range(3)] for i in range(3)])
    far = np.asarray(fit_bt(counts, prior=0))
    assert np.abs(far - spread).max() < 1e-3, (
        f"fed exactly the wins a model with ratings {spread.tolist()} would produce, the fit "
        f"returned {np.round(far, 3).tolist()}. Widely spaced ratings converge slowly: keep "
        "iterating until the ratings stop moving, not for a small fixed number of sweeps"
    )
    print("exercise 2 looks right")


_try("exercise 2", _check_fit_bt)

In [ ]:
def fit_plain(a, b, a_won, k):
    """Votes to ratings in one step: `win_counts` then `fit_bt`. Given to you; the bootstrap
    below refits with it hundreds of times. Each call runs under a time limit, so a fit that
    passed its check but never stops on some other set of votes fails, with a message."""
    with _time_limit(_CALL_SECONDS, "fit_bt (via fit_plain)"):
        return fit_bt(win_counts(a, b, a_won, k))


def _show_a_first_fit() -> None:
    a, b, y = simulate_votes(THETA_TRUE, 12_800, np.random.default_rng(SEED + 2),
                             beta=0.0, p_random=0.0)
    theta_hat = fit_plain(a, b, y, K)
    print(f"{len(y)} careful votes, no position bias, sides randomised")
    print(f"  {'system':<7} {'planted':>8} {'fitted':>8}")
    for i in np.argsort(-THETA_TRUE):
        print(f"  {NAMES[i]:<7} {to_elo(THETA_TRUE)[i]:8.1f} {to_elo(theta_hat)[i]:8.1f}")
    print(f"largest gap between a fitted and a planted rating: "
          f"{np.abs(to_elo(theta_hat) - to_elo(THETA_TRUE)).max():.1f} Elo-like points")


_try("first fit", _show_a_first_fit, needs=("exercise 1", "exercise 2"))

## 5. Two things the arena does to your ratings, and how big they are

The fit above saw only careful voters. Real arenas do not, and the simulator can turn the
two problems on one at a time. The next cell fits the same arena while raising the share of
careless voters, and prints one number for each: the **slope** of the fitted ratings against
the planted ones, computed as `(theta_hat . theta) / (theta . theta)`, where a slope of one
means the scale is recovered exactly and a smaller slope means every gap has shrunk.

A careless voter carries no information, and a coin flip pulls every matchup toward a draw,
so what this measures is by how much. Nothing tells you the slope in a real arena, because
nobody knows the planted strengths; but it tells you what the ratings mean: they are
comparisons on a compressed scale, and the ranking is the part that stays trustworthy.

In [ ]:
def _slope(theta_hat, theta_true):
    """Slope of the fitted ratings on the planted ones (both centred), through the origin."""
    return float(np.dot(theta_hat, theta_true) / np.dot(theta_true, theta_true))


def _show_random_voters() -> None:
    print(f"{'share of random voters':>24} {'slope of fitted on planted':>28}")
    for share in (0.0, 0.15, 0.30):
        a, b, y = simulate_votes(THETA_TRUE, 12_800, np.random.default_rng(SEED + 3),
                                 beta=0.0, p_random=share)
        print(f"{share:>24.2f} {_slope(fit_plain(a, b, y, K), THETA_TRUE):>28.3f}")


_try("random voters", _show_random_voters, needs=("exercise 1", "exercise 2"))

## 6. Exercise 3 — `fit_bt_biased()`: modelling the left-hand bonus

Position bias is a bonus for whoever sits in slot A. If sides are assigned by a fair coin
the bonus lands on every system equally often and mostly washes out of the ratings. If they
are not (the newest model is always on the left, the incumbent always on the right), the
bonus lands on some systems far more than others, and a fit that does not know about it
credits those systems with quality they do not have.

The fix is to give the model a term for it: `P(A wins) = sigmoid(theta_A - theta_B + beta)`.
This is a logistic regression. Each vote is a row of a design matrix with `+1` in the column
of the system in slot A, `-1` in the column of the system in slot B, and a final column of
ones for `beta`. Fit it by Newton's method, which for a logistic loss takes a handful of
iterations. Add a small ridge penalty on every parameter: it stops `beta` running away on
a tiny sample, and, because a shift of all the ratings changes nothing, it also pins the
ratings to mean zero.

<details><summary>💡 Hint 1 — what to think about</summary>

Every vote contributes a row with exactly three non-zero entries: the slot-A system, the
slot-B system and the bias column. Newton needs the gradient and the Hessian of the
penalised negative log-likelihood; the logistic loss has a neat Hessian, and the weights in
it are the predicted variances `p * (1 - p)`.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Build the design matrix once. Start every parameter at zero. Each iteration: predicted
probabilities from the current parameters, gradient equal to the design matrix transposed
times (predicted minus observed) plus the ridge times the parameters, Hessian equal to the
design matrix transposed times itself weighted by the predicted variances plus the ridge on
the diagonal, then subtract the solution of the linear system. Stop when the step is tiny.
Report the ratings shifted to mean zero, and the last parameter as `beta`.
</details>

In [ ]:
def fit_bt_biased(a, b, a_won, k, ridge=1e-3, n_iter=50):
    """Bradley-Terry with a position-bias term, fitted by Newton's method.

    Model: P(slot A wins) = sigmoid(theta[a] - theta[b] + beta). Minimise the negative
    log-likelihood plus `ridge / 2` times the squared length of the parameter vector
    (ratings AND beta), for at most `n_iter` Newton steps, stopping early when the largest
    step is below 1e-10.

    Example:
        >>> theta, beta = fit_bt_biased([0, 1], [1, 0], [1, 1], 2)
        >>> bool(beta > 0)
        True
        (slot A won both votes, whichever system sat in it, so the bias term must be
        positive and the two ratings must come out equal)

    Returns:
        (theta, beta): a length-k array of ratings with mean zero, and the bias as a float.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _expected_votes(theta, beta, pairs, per_pair=4000):
    """Noise-free 'votes': each (a, b) pair repeated so the A-win share is exactly the model's
    probability. A check on the fitting code, not a simulation."""
    aa, bb, yy = [], [], []
    for x, z in pairs:
        p = 1.0 / (1.0 + np.exp(-(theta[x] - theta[z] + beta)))
        wins = int(round(per_pair * p))
        aa += [x] * per_pair
        bb += [z] * per_pair
        yy += [1] * wins + [0] * (per_pair - wins)
    return np.array(aa), np.array(bb), np.array(yy)


def _check_fit_bt_biased() -> None:
    theta = np.array([0.5, -0.2, -0.3])
    both_ways = [(0, 1), (1, 0), (0, 2), (2, 0), (1, 2), (2, 1)]
    a, b, y = _expected_votes(theta, 0.4, both_ways)
    th, be = fit_bt_biased(a, b, y, 3)
    th = np.asarray(th)
    assert th.shape == (3,), f"expected one rating per system, got shape {th.shape}"
    assert abs(be - 0.4) < 0.02, (
        f"with balanced sides the bias should come out near the planted 0.4, got {be:.3f}: "
        "the last parameter is the coefficient of the column of ones"
    )
    assert np.abs(th - (theta - theta.mean())).max() < 0.02, (
        f"ratings should recover the planted ones; got {np.round(th, 3).tolist()}. "
        "Is the slot-B system entered with a MINUS sign in the design matrix?"
    )
    assert abs(th.sum()) < 1e-6, "ratings must be centred: they should sum to zero"
    lazy = [(0, 1), (0, 2), (1, 2)]          # the older system is always in slot A
    a, b, y = _expected_votes(theta, 0.4, lazy)
    th, be = fit_bt_biased(a, b, y, 3)
    assert abs(be - 0.4) < 0.03 and np.abs(np.asarray(th) - (theta - theta.mean())).max() < 0.03, (
        f"with slot A always the lower-numbered system, the fit must still separate the bias "
        f"from the ratings; got beta = {be:.3f} and ratings {np.round(th, 3).tolist()}. "
        "Fit theta and beta together, not one after the other"
    )
    wide = np.array([-2.5, 0.5, 2.0])
    thw, bew = fit_bt_biased(*_expected_votes(wide, 0.9, both_ways, 2000), 3)
    assert np.abs(np.asarray(thw) - wide).max() < 0.05 and abs(bew - 0.9) < 0.05, (
        f"with widely spaced ratings you got {np.round(thw, 3).tolist()} and beta {bew:.3f}: "
        "keep taking Newton steps until the step is tiny, not a fixed few"
    )
    th0, be0 = fit_bt_biased(*_expected_votes(theta, 0.0, both_ways), 3)
    assert abs(be0) < 0.02, f"no planted bias should give a bias near 0, got {be0:.3f}"
    # the fit is defined as the minimiser of a penalised loss, so at the answer its gradient is
    # zero, whatever route the code took to get there (a noisy sample, a strong penalty)
    rng = np.random.default_rng(7)
    ka, na, pen = 3, 200, 8.0
    xa = rng.integers(0, ka, na)
    xb = (xa + rng.integers(1, ka, na)) % ka
    xy = (rng.random(na) < 0.65).astype(int)
    thp, bep = fit_bt_biased(xa, xb, xy, ka, ridge=pen)
    wp = np.append(np.asarray(thp, dtype=float), bep)
    design = np.zeros((na, ka + 1))
    design[np.arange(na), xa] += 1.0
    design[np.arange(na), xb] -= 1.0
    design[:, ka] = 1.0
    grad = design.T @ (1.0 / (1.0 + np.exp(-(design @ wp))) - xy) + pen * wp
    assert np.abs(grad).max() < 1e-4, (
        f"at the answer the gradient of (negative log-likelihood + ridge / 2 * squared length of "
        f"[ratings, beta]) should be zero; yours is {np.round(grad, 4).tolist()}. Is the ridge "
        "in both the gradient and the Hessian, does it cover beta as well as the ratings, and "
        "did the iteration run to convergence?"
    )
    print("exercise 3 looks right")


_try("exercise 3", _check_fit_bt_biased)

In [ ]:
def _show_side_assignment() -> None:
    print(f"planted bias {PLANTED_BETA}, share of random voters {PLANTED_RANDOM}; "
          "12800 votes; ratings compared with the planted ones")
    print("(shape error: root-mean-square distance from the planted ratings after the best "
          "rescaling)")
    print(f"  {'sides':<11} {'fit':<7} {'slope':>6} {'shape error':>12} {'fitted beta':>12}")
    for randomise in (True, False):
        a, b, y = simulate_votes(THETA_TRUE, 12_800, np.random.default_rng(SEED + 4),
                                 PLANTED_BETA, PLANTED_RANDOM, randomise_sides=randomise)
        theta_plain = fit_plain(a, b, y, K)
        with _time_limit(_CALL_SECONDS, "fit_bt_biased"):
            theta_bias, beta_hat = fit_bt_biased(a, b, y, K)
        for name, th in (("plain", theta_plain), ("biased", theta_bias)):
            s = _slope(th, THETA_TRUE)
            err = float(np.sqrt(np.mean((th - s * THETA_TRUE) ** 2)))
            shown = f"{beta_hat:12.3f}" if name == "biased" else f"{'-':>12}"
            print(f"  {'randomised' if randomise else 'lazy':<11} {name:<7} {s:6.3f} {err:12.3f} "
                  f"{shown}")


_try("side assignment", _show_side_assignment,
     needs=("exercise 1", "exercise 2", "exercise 3"))

The table is the reason arenas randomise the slots. Compare the two `plain` rows: the same
voters, the same bias, the same number of votes, and only the slot rule differs. Then compare
the two `lazy` rows: what did modelling the bias change? It is not a licence to stop
randomising: a fit can only undo a bias it has modelled, and a real one depends on the voter,
the prompt and the day. Finally compare the fitted `beta` with the planted one, and think
back to the slope in section 5.

## 7. Exercise 4 — `bootstrap_theta()`: how much would the ratings move on other votes?

A rating is a number computed from one particular list of votes. Had the same voters cast
different votes, you would have got a different list and different ratings. The bootstrap
imitates that: treat the votes you have as the population, draw a new list of the same
length **with replacement**, refit, and repeat. The spread of the refitted ratings is an
honest estimate of how much the ratings depend on which votes you happened to get.

Resample **votes**, not systems, and keep each vote's `(a, b, a_won)` together: a vote is one
observation. `fit` is any function that turns `(a, b, a_won, k)` into a length-`k` vector,
such as `fit_plain`, so this exercise does not depend on the fitting exercises being right.

<details><summary>💡 Hint 1 — what to think about</summary>

One replicate is one list of indices into the votes: as many indices as there are votes, each
one chosen independently and uniformly from all of them, so some votes repeat and some do not
appear. Take the same indices from all three arrays.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Preallocate an array of shape (n_boot, k). Loop n_boot times: draw the indices from the
generator you were given (never from a global source of randomness, or the same seed will
not reproduce the same intervals), index the three arrays with them, call `fit`, store the
result as one row.
</details>

In [ ]:
def bootstrap_theta(a, b, a_won, k, fit, n_boot, rng):
    """Bootstrap replicates of a rating vector, resampling whole votes with replacement.

    Each of the `n_boot` replicates draws `len(a_won)` vote indices uniformly with replacement
    using `rng` (a numpy Generator), takes the matching entries of `a`, `b` and `a_won`
    together, and stores `fit(a_sample, b_sample, won_sample, k)`.

    Example:
        >>> boot = bootstrap_theta(a, b, a_won, 8, fit_plain, 200, np.random.default_rng(1))
        >>> boot.shape
        (200, 8)

    Returns:
        an array of shape (n_boot, k), one row of ratings per replicate.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_bootstrap_theta() -> None:
    n = 60
    a = np.arange(n)
    b = 2 * np.arange(n) + 1
    y = np.arange(n) % 2
    seen = {"rows": [], "bad": 0}

    def spy(sa, sb, sy, k):
        seen["rows"].append(len(sy))
        seen["bad"] += int(np.sum((sb != 2 * sa + 1) | (sy != sa % 2)))
        return np.array([sa.mean(), float(len(np.unique(sa)))])

    boot = bootstrap_theta(a, b, y, 2, spy, 25, np.random.default_rng(3))
    boot = np.asarray(boot)
    assert boot.shape == (25, 2), f"expected shape (n_boot, k) = (25, 2), got {boot.shape}"
    assert seen["rows"] == [n] * 25, (
        f"every replicate must contain as many votes as the original ({n}); yours had sizes "
        f"{sorted(set(seen['rows']))}"
    )
    assert seen["bad"] == 0, (
        "a resampled vote's A-system, B-system and outcome must stay together: draw ONE set of "
        "indices and use it on all three arrays"
    )
    assert boot[:, 1].max() < n, (
        "every replicate had all distinct votes: resampling must be WITH replacement, so some "
        "votes appear twice and others not at all"
    )
    drawn = set()
    bootstrap_theta(a, b, y, 2, lambda sa, sb, sy, k: drawn.update(sa.tolist()) or np.zeros(k),
                    40, np.random.default_rng(5))
    assert drawn == set(range(n)), (
        f"in 40 replicates of {n} votes, votes {sorted(set(range(n)) - drawn)} were never drawn: "
        "every index must be able to come up, the first and the last included (check whether "
        "the upper bound of your random draw is inclusive or exclusive)"
    )
    again = bootstrap_theta(a, b, y, 2, spy, 25, np.random.default_rng(3))
    assert np.array_equal(boot, np.asarray(again)), (
        "the same generator seed must give the same replicates: draw only from `rng`"
    )
    other = bootstrap_theta(a, b, y, 2, spy, 25, np.random.default_rng(4))
    assert not np.array_equal(boot, np.asarray(other)), "different seeds gave identical replicates"
    print("exercise 4 looks right")


_try("exercise 4", _check_bootstrap_theta)

Two helpers, given to you. `percentile_interval` turns the replicates into an interval
for each system. `pair_flip_rates` asks the question a leaderboard reader actually has:
for a given pair of systems, in what share of the replicates does their order differ from
the order your fitted ratings gave? A share near zero means the order survives resampling;
a share near a half means the two are, as far as these votes can tell, a coin flip.

The two answers are not the same. Each rating has its own interval, but the ratings are
computed from the same votes, so their errors are tied together, and two overlapping
intervals do not by themselves mean the order is unstable. The next cell prints both.

In [ ]:
def percentile_interval(theta_boot, level=0.95):
    """Per-system percentile interval of bootstrap replicates: (lower, upper) arrays."""
    tail = 100.0 * (1.0 - level) / 2.0
    return np.percentile(theta_boot, tail, axis=0), np.percentile(theta_boot, 100.0 - tail, axis=0)


def pair_flip_rates(theta_hat, theta_boot, pairs):
    """For each (i, j) in `pairs`: the share of bootstrap replicates in which the order of
    systems i and j differs from their order in `theta_hat`."""
    out = []
    for i, j in pairs:
        ahead = theta_hat[i] > theta_hat[j]
        out.append(float(np.mean((theta_boot[:, i] > theta_boot[:, j]) != ahead)))
    return np.array(out)


def _show_intervals() -> None:
    a, b, y = simulate_votes(THETA_TRUE, 3_200, np.random.default_rng(SEED + 5),
                             PLANTED_BETA, PLANTED_RANDOM)
    theta_hat = fit_plain(a, b, y, K)
    boot = bootstrap_theta(a, b, y, K, fit_plain, 400, np.random.default_rng(SEED + 6))
    lo, hi = percentile_interval(boot)
    rank = list(np.argsort(-theta_hat))
    print(f"{len(y)} votes, {len(boot)} bootstrap replicates, 95% percentile intervals")
    print(f"  {'system':<7} {'rating':>7} {'lower':>7} {'upper':>7}")
    for i in rank:
        print(f"  {NAMES[i]:<7} {to_elo(theta_hat)[i]:7.1f} {to_elo(lo)[i]:7.1f} "
              f"{to_elo(hi)[i]:7.1f}")
    print("\nneighbours in the fitted ranking:")
    print(f"  {'pair':<16} {'intervals overlap':>18} {'share of replicates that swap':>31}")
    pairs = [(rank[r + 1], rank[r]) for r in range(K - 1)]
    flips = pair_flip_rates(theta_hat, boot, pairs)
    for (lower, upper), flip in zip(pairs, flips):
        overlap = bool(lo[upper] < hi[lower])
        print(f"  {NAMES[upper] + ' > ' + NAMES[lower]:<16} {str(overlap):>18} {flip:>31.3f}")


_try("intervals", _show_intervals,
     needs=("exercise 1", "exercise 2", "exercise 4"))

## 8. Exercise 5 — `votes_needed()`: when does the order stop flipping?

An arena is a stream: votes arrive, and the leaderboard is refitted as they do. Take the
first few votes, then twice as many, then twice as many again, and at each size compute, for
one pair of systems, the share of bootstrap replicates that swap them. Plot it against the
number of votes and it falls, but not smoothly, because every size is a different random
dataset.

So the question "how many votes until this pair stops flipping" needs a definition that is
not fooled by one lucky size. This lesson's: the **smallest size from which the flip rate
stays at or below a target for every larger size measured too**. A pair that dips under the
target at one size and bounces back above at the next has not stopped flipping. If even the
largest size is above the target, the honest answer is "not within this stream", which is
`None`, never a guess. An answer equal to the largest size is weaker than it looks: no larger
size was measured, so nothing shows the pair stayed under. Your function still returns it, as
the definition says, and the experiment below marks it as unconfirmed.

<details><summary>💡 Hint 1 — what to think about</summary>

The condition is about a suffix of the list: a size qualifies only if it and everything
after it are at or below the target. Which end of the list is easiest to start from?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Walk the sizes from the largest to the smallest while the flip rate stays at or below the
target, remembering the last size you accepted. The moment you meet one above the target,
stop. If you never accepted anything, there is nothing to report. Watch the comparison at
exactly the target: at the target still counts as stopped.
</details>

In [ ]:
def votes_needed(sizes, flip_rates, target):
    """The smallest vote count from which the flip rate stays <= `target` at every larger size.

    `sizes` is ascending; `flip_rates[i]` is the flip rate measured on the first `sizes[i]`
    votes. A rate exactly equal to `target` counts as within it.

    Example:
        >>> votes_needed([100, 200, 400, 800], [0.30, 0.04, 0.09, 0.02], 0.05)
        800
        (200 dipped under the target but 400 came back above it)

    Returns:
        the size (an element of `sizes`), or None if the largest size is still above `target`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_votes_needed() -> None:
    sizes = [100, 200, 400, 800, 1600]
    got = votes_needed(sizes, [0.30, 0.04, 0.09, 0.03, 0.02], 0.05)
    assert got == 800, (
        f"got {got!r}. 200 is under the target but 400 is back above it, so the order had not "
        "stopped flipping; the answer is the start of the run that lasts to the end"
    )
    got = votes_needed(sizes, [0.30, 0.20, 0.09, 0.07, 0.06], 0.05)
    assert got is None, f"got {got!r}: no size is under the target, so the answer is None"
    got = votes_needed(sizes, [0.30, 0.05, 0.05, 0.05, 0.05], 0.05)
    assert got == 200, (
        f"got {got!r}: a rate exactly equal to the target is within it, so the run starts at 200"
    )
    got = votes_needed(sizes, [0.30, 0.0501, 0.0501, 0.0501, 0.0501], 0.05)
    assert got is None, f"got {got!r}: 0.0501 is above the target of 0.05, however slightly"
    got = votes_needed(sizes, [0.01, 0.01, 0.01, 0.01, 0.01], 0.05)
    assert got == 100, f"got {got!r}: every size is under the target, so the first size is enough"
    got = votes_needed(sizes, [0.01, 0.01, 0.01, 0.01, 0.20], 0.05)
    assert got is None, f"got {got!r}: the largest size is above the target, so the answer is None"
    print("exercise 5 looks right")


_try("exercise 5", _check_votes_needed)

## 9. The experiment: how many votes does each neighbouring pair need?

Everything is ready. The cell below simulates several independent vote streams from the
planted arena, each as long as the largest size. At every size it fits the ratings on the
first votes of the stream, bootstraps them, and measures the flip rate of each pair of
systems that are neighbours in the PLANTED order. (You could not do that in a real arena,
where you do not know the planted order; you would take neighbours in the fitted ranking,
as the intervals table did. Here the planted order lets us compare pairs across sizes.)
Then `votes_needed` reads off the answer for each pair and each stream.

The last column multiplies the votes a pair needed by the square of its planted gap. Sampling
error in a rating shrinks like one over the square root of the number of votes, so if that is
what governs how many votes a pair needs, this column should stay in one neighbourhood for
every pair that resolved. The table prints no verdict on that: judge it yourself from the
column, and remember that a "votes needed" figure is a first-crossing time measured on one
random stream, so some scatter is expected even if the rule holds.

In [ ]:
STREAM_SIZES = [100 * 2 ** i for i in range(10)]
N_STREAMS = 3
N_BOOT = 200
TARGET_FLIP = 0.05
_PLANTED_ORDER = list(np.argsort(THETA_TRUE))
_PLANTED_PAIRS = [(_PLANTED_ORDER[r], _PLANTED_ORDER[r + 1]) for r in range(K - 1)]


def _run_stream(stream: int):
    """Flip rates (sizes x pairs) and count of correctly ordered neighbours (per size)."""
    a, b, y = simulate_votes(THETA_TRUE, STREAM_SIZES[-1],
                             np.random.default_rng([SEED, stream]),
                             PLANTED_BETA, PLANTED_RANDOM)
    flips, right = [], []
    for n in STREAM_SIZES:
        theta_hat = fit_plain(a[:n], b[:n], y[:n], K)
        boot = bootstrap_theta(a[:n], b[:n], y[:n], K, fit_plain, N_BOOT,
                               np.random.default_rng([SEED, stream, n]))
        flips.append(pair_flip_rates(theta_hat, boot, _PLANTED_PAIRS))
        right.append(sum(theta_hat[j] > theta_hat[i] for i, j in _PLANTED_PAIRS))
    return np.array(flips), right


def _shown(v) -> str:
    """A votes-needed answer as the table prints it: None and the largest size are flagged."""
    if v is None:
        return f">{STREAM_SIZES[-1]}"
    return f"{v}?" if v == STREAM_SIZES[-1] else str(v)


def _show_stream_experiment() -> None:
    runs = [_run_stream(s) for s in range(N_STREAMS)]
    print(f"{N_STREAMS} independent streams of {STREAM_SIZES[-1]} votes; {N_BOOT} bootstrap "
          f"replicates per size; target flip rate {TARGET_FLIP}")
    flips0, right0 = runs[0]
    print("\nstream 0: the worst neighbouring pair at each size")
    print(f"  {'votes':>7} {'worst flip rate':>16} {'neighbours in right order':>27}")
    for n, f, r in zip(STREAM_SIZES, flips0.max(axis=1), right0):
        print(f"  {n:>7} {f:>16.3f} {f'{r} of {K - 1}':>27}")
    print("\nvotes needed per neighbouring pair (planted order), one column per stream")
    header = "".join(f"{'stream ' + str(s):>10}" for s in range(N_STREAMS))
    print(f"  {'pair':<16} {'gap':>6}{header} {'votes x gap^2':>14}")
    for p, (i, j) in enumerate(_PLANTED_PAIRS):
        gap = THETA_TRUE[j] - THETA_TRUE[i]
        found = [votes_needed(STREAM_SIZES, runs[s][0][:, p], TARGET_FLIP)
                 for s in range(N_STREAMS)]
        cells = "".join(f"{_shown(v):>10}" for v in found)
        seen = [v for v in found if v is not None and v < STREAM_SIZES[-1]]
        scaled = f"{np.median(seen) * gap ** 2:>14.0f}" if seen else f"{'-':>14}"
        print(f"  {NAMES[i] + ' < ' + NAMES[j]:<16} {gap:6.2f}{cells} {scaled}")
    print(f"  >{STREAM_SIZES[-1]} = still above the target at the largest size; "
          f"{STREAM_SIZES[-1]}? = under it only at the")
    print("  largest size, so unconfirmed and left out of the last column (a median over the "
          "confirmed streams)")
    whole = [votes_needed(STREAM_SIZES, runs[s][0].max(axis=1), TARGET_FLIP)
             for s in range(N_STREAMS)]
    print("\nvotes until EVERY neighbouring pair had stopped flipping: "
          + ", ".join(_shown(v) for v in whole))


_try("stream experiment", _show_stream_experiment,
     needs=("exercise 1", "exercise 2", "exercise 4", "exercise 5"))

Read the table before you read anything into it. Does a pair with a narrow planted gap need
more votes than one with a wide gap? Which pairs did not resolve inside a stream, or resolved
only at the largest size, where nothing confirms it? Do the three streams agree with one
another? The number of votes needed to settle a pair is itself a measurement with noise in
it, and one stream's answer is not the answer.

What this means for an arena that does not know its own truth: publish the interval or the
flip rate next to the rank, so that a rank which has not been measured does not read as one
that has.

## 10. Common mistakes

* **Reading `a_won` as "the first system won" in the wrong place.** In `win_counts` it names
  the slot; it is the winner array you build that names systems. Transposed matrices look
  plausible and quietly reverse the leaderboard.
* **Fitting without a pseudo-count on a small sample.** An undefeated system has an infinite
  maximum-likelihood rating. The fit does not fail; it returns a huge finite number.
* **Trusting a fit that never checked it had stopped.** Fixed-point iteration converges
  slowly on skewed data. Check that the ratings stopped moving, not just that the loop ended.
* **Not randomising the slots.** The biased fit repairs some of the damage. It cannot repair
  a bias it did not model.
* **Resampling systems, or resampling without replacement.** The first changes what you are
  estimating; the second returns the original list in a new order and an interval of width
  zero.
* **Reading overlapping intervals as a tie.** The intervals do not carry the fact that the
  two ratings were estimated from the same votes.
* **Declaring a pair settled at the first size where it dipped under the target.** Measure
  whether it stays there.

In [ ]:
def _show_the_transposed_trap() -> None:
    a, b, y = simulate_votes(THETA_TRUE, 6_400, np.random.default_rng(SEED + 7),
                             PLANTED_BETA, PLANTED_RANDOM)
    w = win_counts(a, b, y, K)
    right = fit_plain(a, b, y, K)
    with _time_limit(_CALL_SECONDS, "fit_bt"):
        wrong = fit_bt(np.asarray(w).T)
    print(f"strongest system by the planted strengths: {NAMES[int(np.argmax(THETA_TRUE))]}")
    print(f"strongest system when the fit reads W as it is:    {NAMES[int(np.argmax(right))]}")
    print(f"strongest system when the fit reads W transposed:  {NAMES[int(np.argmax(wrong))]}")
    print(f"correlation of the two sets of ratings: {np.corrcoef(right, wrong)[0, 1]:.3f}")


_try("transposed matrix", _show_the_transposed_trap, needs=("exercise 1", "exercise 2"))

## 11. Self-check

1. A Bradley-Terry fit returns a rating of 1000 for one system. What does that number tell
   you on its own?
   - (a) The system wins about half of its matchups
   - (b) Nothing on its own: only differences between ratings move a predicted win probability
   - (c) The system has beaten one thousand opponents
   - (d) The system is one thousand points stronger than the weakest one

2. A collector always puts the newest model in slot A, and voters slightly favour slot A.
   The plain fit, which ignores position, will:
   - (a) Be unaffected: a bias toward a slot cannot favour a system
   - (b) Rate the newest models lower, because voters see them less often
   - (c) Fail, because the model is no longer identified
   - (d) Credit the systems shown in slot A more often with quality they may not have

3. A bootstrap replicate is built by:
   - (a) Drawing as many votes as were collected, with replacement, and refitting on them
   - (b) Dropping a random tenth of the systems and refitting
   - (c) Adding random noise to each fitted rating
   - (d) Shuffling the order of the votes and refitting

4. Two systems have overlapping 95% intervals. What is safe to conclude?
   - (a) They are tied, and any ranking between them is meaningless
   - (b) The one with the higher point estimate is better with 95% probability
   - (c) Nothing yet: the share of replicates that swap the pair is the direct measurement
   - (d) The bootstrap failed

5. Under this lesson's assumption that sampling error falls like one over the square root of
   the number of votes, a pair whose planted gap is half as wide as another's needs roughly:
   - (a) Half as many votes
   - (b) The same number of votes
   - (c) Four times as many votes
   - (d) Sixteen times as many votes

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built, and where it goes next

`win_counts` and `fit_bt` are the whole of a plain leaderboard; `fit_bt_biased` is what you
reach for when the collection design is not clean; `bootstrap_theta` and `votes_needed` turn
a ranking into a ranking with a stated amount of doubt, and say how much more data would
remove it. The ranking uncertainty measured here is the quantity a later lesson on how many
evaluation items are enough needs as its starting point.

What this lesson left out on purpose: ties, several voters with different reliabilities,
adaptive choice of which pair to show next, and any correction for answer style or length.
Each is a real complication of a live arena, and none can be studied honestly until you can
measure the simple case, which is what you just did.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_win_counts, ()),
        ("exercise 2", _check_fit_bt, ()),
        ("exercise 3", _check_fit_bt_biased, ()),
        ("exercise 4", _check_bootstrap_theta, ()),
        ("exercise 5", _check_votes_needed, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.